[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/humanization/humanization_guide/05_humanize_sapiens/05_sapiens_lab.ipynb)


# 05 — BioPhi/Sapiens — 후보 생성 · CDR 사고 · humanness

> 본문 [`05_humanize_sapiens.md`](05_humanize_sapiens.md) 와 **한 절씩 짝지어** 보세요.
> **실측 소요 —** Sapiens `predict_scores` VH **0.93초** / VL **0.53초** · humanized 재스코어링 **0.01초**
> **앞 랩에서 이어져요** — Ch.04 numbering 의 `my_run/` 을 먼저 찾고, 없으면 커밋된 `data/` 로 대신합니다.

## 0) Colab 준비 — 저장소 클론 & 작업 폴더 이동

이 셀이 저장소를 클론하고 `05_humanize_sapiens/` 로 이동해 필요한 패키지만 깝니다(로컬에서 `05_humanize_sapiens/` 안에 열었다면 클론 생략).
끝나면 **`my_run/`**(내가 만들 결과)과 **`data/`**(커밋된 레퍼런스)가 준비돼요 — 랩은 `my_run/` 을 먼저 찾고 없으면 `data/` 로 폴백하며 어느 쪽을 썼는지 매번 print 합니다.

In [ ]:
# ====== Colab/로컬 공용 부트스트랩 (모든 챕터 공통) ======
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # 이 가이드 저장소 (fork 했다면 본인 주소로 바꾸세요)
CLONE_AS = "bio-guides"
CHAPTER  = "05_humanize_sapiens"
APT_PKGS = ""     # Colab 에서만: 시스템 패키지 (hmmer = ANARCI 가 부르는 hmmscan)
PIP_PKGS = "pandas matplotlib sapiens"     # 없는 것만 설치

import os, sys, json, time, shutil, pathlib, subprocess, importlib, importlib.util
IN_COLAB = "google.colab" in sys.modules

# 라이브러리 내부 소음 끄기 — 아래 두 줄은 결과와 무관한데 셀 출력 첫머리를 덮어요.
#   · tqdm "IProgress not found"  : 진행바를 위젯 대신 글자로 그린다는 안내일 뿐
#   · pkg_resources is deprecated : lightning/torch 가 남긴 setuptools 경고
# 진단이 필요하면 이 두 줄을 지우고 다시 실행하세요.
import warnings
warnings.filterwarnings("ignore", message=".*IProgress not found.*")
warnings.filterwarnings("ignore", message=".*pkg_resources is deprecated.*")
warnings.filterwarnings("ignore", message=".*Bio.pairwise2 has been deprecated.*")
# 가중치 로딩 진행바와 transformers 안내문도 결과를 밀어내요 (import 전에 꺼 둡니다)
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
import logging
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)   # "unauthenticated requests" 안내

# HuggingFace 가중치 다운로드가 '멈춘 채' 매달리는 일을 막아요.
# (멈춤은 예외가 안 나서 폴백이 안 걸려요 — 타임아웃을 걸어 실패로 바꿔야 data/ 로 이어져요)
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")   # 스트림 30초 무응답 → 끊고 재시도
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "15")

def _run(cmd, check=True, timeout=None, quiet=False):
    # timeout 을 꼭 주세요 — 네트워크가 '멈춘 채' 매달리면 예외가 안 나서 data/ 폴백이 안 걸립니다.
    """quiet=True 면 출력을 삼키고 **실패했을 때만** 보여줘요.
    apt-get 은 "(Reading database ... 5%(Reading database ... 10%" 같은 진행률을 600자 넘게 쏟아내는데,
    그게 노트북을 연 학습자가 보는 첫 화면을 덮어버려서 실패로 오해하게 만들거든요."""
    print("$", cmd)
    if not quiet:
        return subprocess.run(cmd, shell=True, check=check, timeout=timeout)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    if p.returncode != 0:
        print((p.stdout or "") + (p.stderr or ""))
        if check:
            raise subprocess.CalledProcessError(p.returncode, cmd)
    return p

_MARK = "humanization_viz.py"          # 이 파일이 있는 폴더 = 가이드 루트

def _find_root():
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    # 클론 직후: cwd '아래'만 깊이 3까지 훑는다.
    # (상위로 올라가 rglob 하면 Colab 에서는 / 전체를 뒤지게 된다 — 매우 느리고 엉뚱한 경로를 잡는다)
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "가이드 루트를 못 찾았어요. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

GUIDE_ROOT = ROOT.resolve()
os.chdir(GUIDE_ROOT / CHAPTER)         # data/ 상대경로가 그대로 동작
sys.path.insert(0, str(GUIDE_ROOT))    # humanization_viz import
sys.path.insert(0, str(GUIDE_ROOT / CHAPTER))

_IMPORT_NAME = {"biopython": "Bio", "pyyaml": "yaml", "scikit-learn": "sklearn"}

def _have(mod):
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False

def _ensure(pkgs):
    miss = [p for p in pkgs.split() if not _have(_IMPORT_NAME.get(p, p.replace("-", "_")))]
    if miss:
        print("설치:", " ".join(miss))
        _run(f'"{sys.executable}" -m pip -q install ' + " ".join(miss))
        importlib.invalidate_caches()

_APT = APT_PKGS.split() if (APT_PKGS and IN_COLAB) else []   # 모아서 apt 를 한 번만 돌립니다
if PIP_PKGS:
    _ensure(PIP_PKGS)

def _ensure_pkg_resources():
    # setuptools 81+(2026-02) 이 pkg_resources 를 없앴는데 IgFold 의존성이 이걸 import 합니다.
    if importlib.util.find_spec("pkg_resources") is None:
        _run(f'"{sys.executable}" -m pip -q install "setuptools<81"')
        importlib.invalidate_caches()

import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    _APT.append("fonts-nanum")             # Colab 엔 한글 폰트가 없어 라벨이 □ 로 깨져요

if _APT:                                   # apt 인덱스 갱신은 한 번만 (ANARCI 는 hmmscan 이 필요해요)
    _run("apt-get -qq update", timeout=600, quiet=True)
    _run("DEBIAN_FRONTEND=noninteractive apt-get -qq install -y " + " ".join(_APT), timeout=900, quiet=True)


# ── ① 내가 만든 결과(my_run) 우선 · ② 없으면 커밋된 레퍼런스(data/) ─────────────
MY  = pathlib.Path("my_run"); MY.mkdir(exist_ok=True)
REF = pathlib.Path("data")

def find_one(pattern, ref=REF, quiet=False):
    """산출물 하나 찾기 — 내가 만든 my_run/ 을 먼저 뒤지고, 없으면 커밋된 레퍼런스에서."""
    hits = sorted(MY.rglob(pattern))
    if hits:
        if not quiet: print(f"[내 결과]   {hits[0]}")
        return hits[0]
    hits = sorted(pathlib.Path(ref).glob(pattern))
    assert hits, f"'{pattern}' 을 my_run/ 에서도 {ref}/ 에서도 찾지 못했어요."
    if not quiet: print(f"[레퍼런스] {hits[0]}")
    return hits[0]

def find_prev(chapter, pattern, ref=REF, quiet=False):
    """앞 챕터에서 직접 만든 산출물 → 없으면 이 챕터 data/ 레퍼런스."""
    hits = sorted((GUIDE_ROOT / chapter / "my_run").rglob(pattern))
    if hits:
        if not quiet: print(f"[내 결과 · {chapter}] {hits[0]}")
        return hits[0]
    return find_one(pattern, ref, quiet)

def read_fasta(path):
    out, name, buf = {}, None, []
    for line in pathlib.Path(path).read_text().splitlines():
        if line.startswith(">"):
            if name: out[name] = "".join(buf)
            name, buf = line[1:].strip(), []
        elif line.strip():
            buf.append(line.strip())
    if name: out[name] = "".join(buf)
    return out

def write_fasta(path, records):
    pathlib.Path(path).write_text("".join(f">{k}\n{v}\n" for k, v in records.items()))
    return pathlib.Path(path)

PARENTAL = read_fasta(REF / "parental.fasta")      # 가이드 전체를 관통하는 러닝 예제
VH = PARENTAL["parental_H"]
VL = PARENTAL["parental_L"]

print("가이드 루트 :", GUIDE_ROOT)
print("작업 폴더   :", pathlib.Path.cwd())
print(f"러닝 예제   : VH {len(VH)} aa · VL {len(VL)} aa (mouse hybridoma 가정)")

## 1) 직접 실행 — 설치 함정을 넘고 Sapiens 돌리기 (본문 5.1~5.2)

첫 줄에서 막히는 분이 많아요. `pip install biophi` 는 이렇게 죽어요.

```
ERROR: Could not find a version that satisfies the requirement biophi (from versions: none)
```

BioPhi 는 **bioconda 전용**(v1.0.11) 이라 PyPI 에 아예 없어요. 반면 사람화를 실제로 담당하는 엔진 **Sapiens 는 PyPI 에 따로**(`sapiens` 1.1.0) 올라와 있어요. 후보 서열만 빨리 뽑을 거면 이쪽이 제일 빠른 길이고, 0) 부트스트랩이 깐 것도 이거예요.

```bash
conda install -c bioconda biophi     # 전체(웹·CLI 포함) — PyPI 아님
python -m pip install sapiens        # humanization 엔진만
```

핵심 함수는 `predict_scores` 하나예요 — position 마다 20개 아미노산에 대한 **사람 모델의 확률 분포**를 주고, 각 자리에서 확률이 가장 높은 잔기를 이어 붙이면(**argmax**) 그게 Sapiens-humanized 서열이에요. 모델 가중치는 첫 실행 때 HuggingFace 에서 받아와요.

**가드 없이** 돌립니다 — 본문이 경고한 사고를 직접 재현하려고요.

In [ ]:
import pandas as pd, numpy as np
from IPython.display import display

def mutations(par, hum):
    """길이가 같은 두 서열의 치환 목록 (raw 1-based)."""
    return [{"position_1based": i + 1, "wt": a, "mut": b, "mutation": f"{a}{i+1}{b}"}
            for i, (a, b) in enumerate(zip(par, hum)) if a != b]

RAN_SAPIENS, mat_h, mat_l = False, None, None
# 부트스트랩이 HF_HUB_DOWNLOAD_TIMEOUT=30 을 걸어 '멈춤'을 실패로 바꿔 뒀어요.
# 여기서는 두 번까지 재시도하고, 그래도 안 되면 커밋된 실행 산출물(data/)로 이어가요.
for _attempt in (1, 2):
    if not _have("sapiens"):
        print("sapiens 모듈이 없어요 — python -m pip install sapiens 로 깔면 이 셀이 실제로 돌아요.")
        break
    try:
        import sapiens
        t0 = time.time()
        mat_h = sapiens.predict_scores(VH, "H")          # rows=position, cols=20 AA
        mat_l = sapiens.predict_scores(VL, "L")
        hum_h = "".join(mat_h.columns[mat_h.values.argmax(axis=1)])   # 가드 없는 argmax
        hum_l = "".join(mat_l.columns[mat_l.values.argmax(axis=1)])
        print(f"Sapiens predict_scores VH+VL {time.time()-t0:.2f}초")
        mat_h.to_csv(MY / "score_matrix_VH_parental.csv", index_label="position0based")
        mat_l.to_csv(MY / "score_matrix_VL_parental.csv", index_label="position0based")
        write_fasta(MY / "sapiens_humanized_noguard.fasta",
                    {"sapiens_humanized_VH": hum_h, "sapiens_humanized_VL": hum_l})
        RAN_SAPIENS = True
        break
    except Exception as e:
        print(f"Sapiens 실행 실패({_attempt}/2)", type(e).__name__, str(e)[:160])
        if _attempt == 1:
            print("  · HuggingFace 다운로드 지연일 수 있어 5초 뒤 다시 시도해요")
            time.sleep(5)

if not RAN_SAPIENS:
    f = read_fasta(REF / "sapiens_humanized.fasta")
    hum_h, hum_l = f["sapiens_humanized_VH"], f["sapiens_humanized_VL"]
    mat_h = pd.read_csv(REF / "score_matrix_VH_parental.csv", index_col=0)
    mat_l = pd.read_csv(REF / "score_matrix_VL_parental.csv", index_col=0)
SRC = "내 실행(my_run/)" if RAN_SAPIENS else "레퍼런스(data/)"

# mutation 은 어느 경로든 **서열에서 직접 다시** 셉니다 — 6) 의 대조가 진짜 대조가 되도록.
mut_h = pd.DataFrame(mutations(VH, hum_h))
mut_l = pd.DataFrame(mutations(VL, hum_l))
if RAN_SAPIENS:
    mut_h.to_csv(MY / "mutations_VH.csv", index=False)
    mut_l.to_csv(MY / "mutations_VL.csv", index=False)
    print("→", MY / "sapiens_humanized_noguard.fasta", "· mutations_VH.csv · mutations_VL.csv")

def show_pair(tag, par, hum, mut, width=60):
    """긴 서열을 60자씩 끊어 위아래로 세우고, 바뀐 자리 아래에 ^ 를 찍어요.
    한 줄로 120자를 쭉 흘리면 어디가 바뀌었는지 눈으로 못 찾거든요."""
    diff = {int(m[1:-1]) for m in mut["mutation"]} if len(mut) else set()
    print(f"\n[{tag}] parental → sapiens · {len(diff)}자리 변경 (^ 가 바뀐 자리)")
    for s in range(0, len(par), width):
        e = min(s + width, len(par))
        print(f"  {s+1:>4d} parental {par[s:e]}")
        print(f"       sapiens  {hum[s:e]}")
        print(" " * 16 + "".join("^" if (i + 1) in diff else " " for i in range(s, e)))

for tag, par, hum, mut in (("VH", VH, hum_h, mut_h), ("VL", VL, hum_l, mut_l)):
    show_pair(tag, par, hum, mut)

mut_all = pd.concat([mut_h.assign(체인="VH"), mut_l.assign(체인="VL")], ignore_index=True)
if len(mut_all):
    print("\n바뀐 자리 목록 (raw 1-based)")
    display(mut_all.rename(columns={"position_1based": "raw 위치", "wt": "원본",
                                    "mut": "변경", "mutation": "표기"})
            [["체인", "raw 위치", "원본", "변경", "표기"]])
else:
    print("\n바뀐 자리가 하나도 없어요 — argmax 가 parental 을 그대로 돌려준 거예요.")

print("서열 출처 —", SRC)
print(f"판정 — VH {len(mut_h)}/{len(VH)} 자리가 바뀌어 parental 대비 identity {1-len(mut_h)/len(VH):.1%}, "
      f"VL 은 {len(mut_l)}/{len(VL)} 로 {1-len(mut_l)/len(VL):.1%} 예요. "
      "네 줄짜리 argmax 가 서열을 이만큼 갈아엎었어요 — 어디를 갈아엎었는지는 다음 절에서 봅니다.")

## 2) 내 결과 확인 — CDR 이 어떻게 됐나 (본문 5.3)

`argmax` 는 **CDR 인지 framework 인지 구분하지 않아요.** 모델은 "사람 항체에서 이 자리에 뭐가 자주 오는가"만 알지, "이 항체가 무엇에 붙어야 하는가"는 모르니까요.

그러니 개수만 세면 안 되고 **자리**를 봐야 해요. Ch.04 에서 못 박은 CDR 좌표를 가져와 6개 CDR 을 하나씩 before → after 로 펼쳐 봅니다. Ch.04 를 건너뛰었다면 이 챕터 `data/cdr_table_imgt.csv` 로 폴백해요 — 이 노트북은 항상 내 결과를 먼저 찾고, 없으면 커밋된 레퍼런스로 이어가요.

In [ ]:
ct = pd.read_csv(find_prev("04_sequence_qc", "cdr_table_imgt.csv"))
have_raw = {"raw_start", "raw_end"}.issubset(ct.columns)     # Ch.04 my_run 표에만 있는 좌표 컬럼

cdr_rows, guard, hits_all = [], {"H": [], "L": []}, {"H": [], "L": []}
for _, r in ct.iterrows():
    chain = r["chain"]
    par, hum = (VH, hum_h) if chain == "H" else (VL, hum_l)
    if have_raw and pd.notna(r["raw_start"]):
        st, en = int(r["raw_start"]), int(r["raw_end"])
    else:
        st = par.find(r["sequence"]) + 1          # 못 찾으면 0 이 되니 바로 아래에서 걸러요
        en = st + len(r["sequence"]) - 1
    if st < 1 or en > min(len(par), len(hum)):
        print(f"  {chain}-{r['cdr']} 좌표를 못 잡아 건너뜁니다(입력 서열이 레퍼런스와 다르면 생겨요)")
        continue
    rng = list(range(st, en + 1))
    hits = [f"{par[p-1]}{p}{hum[p-1]}" for p in rng if par[p-1] != hum[p-1]]
    guard[chain] += rng
    hits_all[chain] += hits
    cdr_rows.append({"CDR": f"{chain}-{r['cdr']}", "raw 좌표": f"{st}..{en}",
                     "parental": "".join(par[p-1] for p in rng),
                     "Sapiens":  "".join(hum[p-1] for p in rng),
                     "변이": ", ".join(hits) or "—", "파손": bool(hits)})

cdr = pd.DataFrame(cdr_rows)
display(cdr)

gp = GUIDE_ROOT / "04_sequence_qc" / "my_run" / "cdr_guard.json"
if gp.exists():
    prev = json.loads(gp.read_text())
    ok = all(sorted(set(prev.get(k, []))) == sorted(set(guard[k])) for k in ("H", "L"))
    print("Ch.04 cdr_guard.json 좌표와 일치 —", ok)

n_mut = len(mut_h) + len(mut_l)
n_cdr = len(hits_all["H"]) + len(hits_all["L"])
broken = cdr[cdr["파손"]]
print(f"\nCDR 안에 떨어진 변이 — VH {len(hits_all['H'])}개 · VL {len(hits_all['L'])}개 "
      f"(전체 {n_mut}개 중 {n_cdr}개, {n_cdr/n_mut:.0%})")
print(f"바뀐 CDR {len(broken)}/{len(cdr)}개 —", ", ".join(broken["CDR"]) or "없음")
h3 = cdr[cdr["CDR"] == "H-CDR3"]
if len(h3):
    r3 = h3.iloc[0]
    print(f"CDR-H3   {r3['parental']} → {r3['Sapiens']}   ({r3['변이']})")
    print("판정 — Ch.04 가 '여기 변이가 들어가면 빨간불' 이라고 못 박은 그 loop 예요. " +
          ("항원과 직접 만나는 자리가 바뀌었으니 사람다움과 별개로 결합력을 반드시 확인해야 해요."
           if r3["파손"] else "여기는 그대로 보존됐어요."))
print("그래서 실무에서는 셋 중 하나를 꼭 해요 — ① CDR 좌표를 argmax 대상에서 제외, "
      "② 도구의 CDR 보호 모드, ③ 후처리로 CDR 을 parental 로 되돌리기.")

## 3) 직접 실행 — CDR 가드 적용본 만들기 (본문 5.3 주의)

셋 중 **③ 후처리 복원**을 그대로 구현해요. 가장 단순하고, 검증하기도 제일 쉬워요 — CDR 좌표의 잔기를 parental 로 되돌리기만 하면 되니까요.

In [ ]:
def cdr_guarded(par, hum, protected):
    """protected(raw 1-based) 자리의 잔기를 parental 로 되돌린 서열."""
    s = list(hum)
    for p in protected:
        s[p - 1] = par[p - 1]
    return "".join(s)

g_h = cdr_guarded(VH, hum_h, guard["H"])
g_l = cdr_guarded(VL, hum_l, guard["L"])
write_fasta(MY / "sapiens_humanized_cdrguard.fasta",
            {"sapiens_guarded_VH": g_h, "sapiens_guarded_VL": g_l})

cmp_rows = []
for tag, par, ng, gd, pr in (("VH", VH, hum_h, g_h, guard["H"]), ("VL", VL, hum_l, g_l, guard["L"])):
    cmp_rows.append({"체인": tag, "보호 좌표": len(pr),
                     "가드 없음 · 총 변이": sum(a != b for a, b in zip(par, ng)),
                     "가드 없음 · CDR 변이": sum(par[p-1] != ng[p-1] for p in pr),
                     "가드 적용 · 총 변이": sum(a != b for a, b in zip(par, gd)),
                     "가드 적용 · CDR 변이": sum(par[p-1] != gd[p-1] for p in pr)})
cg = pd.DataFrame(cmp_rows)
display(cg)

left = int(cg["가드 적용 · CDR 변이"].sum())
print("→", MY / "sapiens_humanized_cdrguard.fasta")
print(f"판정 — 가드 적용 뒤 CDR 변이 {left}개. " +
      ("0 이면 CDR 은 parental 그대로이고, framework 치환만 남아요."
       if left == 0 else "0 이 아니면 보호 좌표가 서열과 어긋난 거예요 — Ch.04 의 CDR 표부터 다시 보세요.") +
      f" 총 변이는 VH {int(cg.loc[0, '가드 없음 · 총 변이'])}→{int(cg.loc[0, '가드 적용 · 총 변이'])}, "
      f"VL {int(cg.loc[1, '가드 없음 · 총 변이'])}→{int(cg.loc[1, '가드 적용 · 총 변이'])} 로 줄어요.")

## 4) humanness — **정의를 못 박고** 계산 (본문 5.4)

변이를 21개 넣었다고 사람다워졌다는 보장은 없어요. 숫자로 확인해야죠. 그런데 "humanized 의 humanness" 는 계산법이 두 가지고 **값이 서로 달라요.**

| 정의 | 계산 방식 | 이 실행에서 |
|---|---|---:|
| **(a) argmax-on-parental** | parental 문맥 행렬의 position 별 **최대 확률** 평균 — humanized 서열을 다시 스코어링하진 않아요 | 0.782 / 0.851 |
| **(b) 재스코어링 self-prob** | humanized 서열을 `predict_scores` 에 **다시 넣어** 그 서열 **자기 잔기**의 확률 평균 | **0.815 / 0.872** |

본문 표·그림의 값은 **(b)** 예요. 둘 다 직접 계산해서 눈으로 확인해요.

In [ ]:
def mean_self_prob(mat, seq):
    """확률행렬(rows=position, cols=20 AA)에서 서열 자기 잔기 확률의 평균."""
    return float(np.mean([mat.iloc[i][aa] for i, aa in enumerate(seq)]))

rows, HAVE_GUARD_COL = [], RAN_SAPIENS
for tag, par, ng, gd, chain, mat in (("VH", VH, hum_h, g_h, "H", mat_h),
                                     ("VL", VL, hum_l, g_l, "L", mat_l)):
    row = {"chain": tag,
           "parental": round(mean_self_prob(mat, par), 4),                     # parental self-prob
           "(a) argmax-on-parental": round(float(np.mean(mat.values.max(axis=1))), 4)}
    if RAN_SAPIENS:
        row["(b) 재스코어링 humanized"] = round(mean_self_prob(sapiens.predict_scores(ng, chain), ng), 4)
        row["(b) CDR 가드 적용본"]      = round(mean_self_prob(sapiens.predict_scores(gd, chain), gd), 4)
    else:
        # 커밋된 재스코어링 행렬이 있어 (b) 는 그대로 계산돼요. 가드 적용본은 그 서열을
        # predict_scores 에 다시 넣어야 나오는 값이라, 실행 없이는 열을 만들지 않아요.
        rs = pd.read_csv(REF / f"score_matrix_{tag}_humanized_rescored.csv", index_col=0)
        row["(b) 재스코어링 humanized"] = round(mean_self_prob(rs, ng), 4)
    rows.append(row)

hn = pd.DataFrame(rows)
if RAN_SAPIENS:
    hn.to_csv(MY / "humanness_summary.csv", index=False)
display(hn)

if not HAVE_GUARD_COL:
    print("CDR 가드 적용본 열은 비워 두지 않고 아예 만들지 않았어요 — 그 서열을 predict_scores 에 "
          "다시 넣어야 나오는 값이라, 1) 셀에서 Sapiens 가 실제로 돌면 그때 생겨요.")

for _, r in hn.iterrows():
    line = (f"{r['chain']}  parental {r['parental']:.3f} → (b) {r['(b) 재스코어링 humanized']:.3f} "
            f"(▲ +{r['(b) 재스코어링 humanized'] - r['parental']:.3f})  ·  "
            f"(a) {r['(a) argmax-on-parental']:.3f} · (a)↔(b) 차 "
            f"{abs(r['(b) 재스코어링 humanized'] - r['(a) argmax-on-parental']):.3f}")
    if HAVE_GUARD_COL:
        line += (f"  ·  가드 적용본 {r['(b) CDR 가드 적용본']:.3f} "
                 f"(▼ {r['(b) 재스코어링 humanized'] - r['(b) CDR 가드 적용본']:.3f})")
    print(line)

print("판정 — 같은 실행인데 (a) 와 (b) 가 다른 건 어느 하나가 틀려서가 아니라 재는 대상이 달라서예요. "
      "humanness 를 보고할 때 정의를 반드시 밝혀야 하는 이유예요.")
if HAVE_GUARD_COL:
    drop = (hn["(b) 재스코어링 humanized"] - hn["(b) CDR 가드 적용본"]).max()
    print(f"가드 적용본은 (b) 보다 최대 {drop:.3f} 낮아요 — CDR 을 사람화하지 않았으니 당연하고, "
          "그게 결합력을 지키는 대가예요.")
print("이 값은 Sapiens 모델의 확률이지 OASis 점수가 아니에요 — OASis 는 서열을 9-mer 로 쪼개 "
      "OAS DB 에서 얼마나 자주 관찰되는지로 매기는 별개 지표이고 DB 다운로드가 필요해요(본문 5.4 심화).")

## 5) 그래프 — parental vs humanized (본문 5.4)

본문 그림과 같은 막대를 공용 모듈 `humanization_viz.humanness_bars` 로 그려요. 저장 위치는 `my_run/` 이에요 — 챕터 폴더의 `05_humanness.png` 는 본문이 인용하는 커밋 파일이라 건드리지 않아요.

In [ ]:
from humanization_viz import humanness_bars
from IPython.display import Image, display

bars = [{"chain": r["chain"], "parental": r["parental"],
         "humanized": r["(b) 재스코어링 humanized"]} for _, r in hn.iterrows()]
png = humanness_bars(bars, "Sapiens humanness — parental vs humanized (정의 b)",
                     MY / "05_humanness.png")
display(Image(str(png)))

pv = {r["chain"]: (r["parental"], r["(b) 재스코어링 humanized"]) for _, r in hn.iterrows()}
low  = min(pv, key=lambda k: pv[k][0])
gain = {k: v[1] - v[0] for k, v in pv.items()}
big  = max(gain, key=gain.get)
print(f"출발점이 낮은 쪽은 {low} ({pv[low][0]:.3f}), 상승폭이 큰 쪽은 {big} (+{gain[big]:.3f}) 이에요.")

glp = GUIDE_ROOT / "04_sequence_qc" / "data" / "germline_assignment.csv"
if glp.exists():
    gv = pd.read_csv(glp)
    gv = gv[gv["gene_type"] == "V"].set_index("chain")["identity_pct"]
    if {"H", "L"}.issubset(gv.index):
        gl_low = "VH" if gv["H"] < gv["L"] else "VL"
        print(f"Ch.04 의 V germline identity 는 VH {gv['H']:.2f}% · VL {gv['L']:.2f}% 로 낮은 쪽이 {gl_low} 였어요.")
        print("판정 — 두 지표가 " + ("같은" if gl_low == low else "다른") +
              " 체인을 가리켜요. " + ("germline 거리와 Sapiens 확률이 같은 결론을 주면 그 판단은 믿고 갑니다."
                                    if gl_low == low else "두 지표가 갈리면 어느 축으로 우선순위를 매길지 먼저 정하세요."))
else:
    print("판정 — 출발점이 낮은 체인이 더 크게 오르는 게 정상이에요. 이미 사람다운 체인은 올릴 여지가 적어요.")

## 6) 레퍼런스 대조

`data/` 는 이 가이드를 만들 때 **실제로 돌려 나온** 산출물이에요. 변이 목록이 문자 단위로 같은지, humanness 가 정의별로 같은지 맞춰 봅니다.

In [ ]:
for tag, mine in (("VH", mut_h), ("VL", mut_l)):
    ref = pd.read_csv(REF / f"mutations_{tag}.csv")
    a, b = list(mine["mutation"]), list(ref["mutation"])
    print(f"{tag} — 내 계산 {len(a)}개 / 레퍼런스 {len(b)}개 · 목록 일치 {a == b}")
    if a != b:
        print("   내 계산에만 :", ", ".join(sorted(set(a) - set(b))) or "—")
        print("   레퍼런스에만:", ", ".join(sorted(set(b) - set(a))) or "—")
print("(서열 출처는", SRC + " 이고, 변이 목록은 그 서열에서 다시 세어 data/mutations_*.csv 와 맞춘 거예요.)")

ref_hn = pd.read_csv(REF / "humanness_summary.csv")
DEF_KEY = {"parental": "parental",
           "(a) argmax-on-parental": "definition_a_argmax_on_parental_matrix",
           "(b) 재스코어링 humanized": "definition_b_rescored_humanized"}
rows = []
for _, r in hn.iterrows():
    for col, key in DEF_KEY.items():
        if col not in hn.columns:
            continue
        sel = ref_hn[(ref_hn["chain"] == r["chain"]) & (ref_hn["definition"] == key)]
        if not len(sel):
            print(f"  레퍼런스에 {r['chain']} / {key} 행이 없어 이 줄은 건너뜁니다")
            continue
        v, w = float(r[col]), float(sel["mean_probability"].iloc[0])
        rows.append({"chain": r["chain"], "정의": col, "내 결과": round(v, 4),
                     "레퍼런스": round(w, 4), "차": round(abs(v - w), 4)})
chk = pd.DataFrame(rows)
display(chk)
mx = float(chk["차"].max())
print(f"판정 — 정의별 최대 차이 {mx:.4f}. " +
      ("1e-3 아래면 같은 실행을 그대로 재현한 거예요." if mx < 1e-3 else
       "1e-3 을 넘으면 모델 버전이나 입력 서열이 다른 거예요 — sapiens 버전부터 확인하세요."))

## 이 랩에서 확인한 것

1. **설치** — BioPhi 는 **bioconda**(v1.0.11), Sapiens 는 **PyPI `sapiens`**(1.1.0). `pip install biophi` 는 실패해요.
2. **Sapiens humanization = `predict_scores` 의 position 별 argmax.** 실측 **VH 21 변이 · VL 17 변이**(identity 82.5% / 84.7%).
3. **argmax 는 CDR 을 구분하지 않아요** — 6개 CDR 중 **5개**가 바뀌었어요. `H-CDR2`(Y52N·N58T), `H-CDR3`(L104D·Y109V), `L-CDR1`(H31A·K32Y·F33N·P34D), `L-CDR2`(K52G·L54S), `L-CDR3`(R98S). 전체 38개 변이 중 **11개가 CDR 안**이에요.
4. **CDR-H3 `ARRGRYGLYAMDY` → `ARRGRYGDYAMDV`** — Ch.04 가 빨간불이라고 못 박은 loop 에 변이 2개가 들어갔어요. 후처리 복원(`sapiens_humanized_cdrguard.fasta`)을 거치면 CDR 변이 **0개**, 총 변이는 VH 21→17 · VL 17→10 이에요.
5. **humanness 는 정의를 밝혀야** 해요. **(b) 재스코어링** 기준 VH **0.694 → 0.815** · VL **0.770 → 0.872**, 같은 실행을 **(a)** 로 계산하면 **0.782 / 0.851** 이에요. 가드 적용본은 (b) 보다 조금 낮고, 그게 결합력을 지키는 대가예요.
6. 이 값은 Sapiens 모델 확률이지 **OASis 가 아니에요.** OASis 는 9-mer 를 OAS DB 와 대조하는 별개 지표예요.
7. 산출물 — `my_run/sapiens_humanized_noguard.fasta` · `sapiens_humanized_cdrguard.fasta` · `mutations_VH.csv` · `mutations_VL.csv` · `humanness_summary.csv` · `05_humanness.png`. Ch.06~Ch.10 이 이 파일들을 그대로 씁니다.

다음 → **[Ch.06 — Humatch · AnthroAb](../06_cdr_safe_tools/06_tools_lab.ipynb)**